In [1]:
from torchvision.datasets import CelebA
from torchvision import transforms

from torchvision.models import ResNet18_Weights

import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import random_split


from torchmetrics.classification import BinaryAUROC, Accuracy

import torch
from pytorch_lightning.loggers import WandbLogger

import numpy as np 

import wandb

from datetime import datetime
from tqdm.notebook import tqdm

from collections import OrderedDict

from xaikd import utils, attributors, models

import pandas as pd
from xaikd.bases import PRCAReconGreedy

from matplotlib import pyplot as plt 

import numpy.typing as npt

import os

In [2]:
DATA_ROOT = "../../datasets"

WANDB_PROJECT = "xaikd-training-teacher-models"
WANDB_GROUP = "celeba"

NUM_WORKERS = 16
BATCH_SIZE = 64
NUM_ATTRIBUTES = 40

TRAINING_SIZE = 0.1

SEED = 1

DEVICE = "cuda"

ARCH = "resnet18"

RUN_ID = "n8r0q2vb"
# RUN_ID = "6oj5aaxl" # imagenet pretrained
# RUN_ID = "dskgwbyk" # fc.bias= False


# WANDB_PROJECT = "kitchen-sink"
# RUN_ID = "jtp7uv29"

In [3]:
TRANSFORMATION_DEFAULT = ResNet18_Weights.IMAGENET1K_V1.transforms()

In [4]:
TRANSFORMATION_DEFAULT.mean, TRANSFORMATION_DEFAULT.std

([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

In [5]:
ds_train = CelebA(
    root=DATA_ROOT, split="train", target_type="attr",
    transform=TRANSFORMATION_DEFAULT
)

trng = torch.Generator()
trng.manual_seed(1)
ds_train, _ = random_split(ds_train, [TRAINING_SIZE, 1-TRAINING_SIZE], generator=trng)

dl_train = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,    
    shuffle=False,
)

In [6]:
ds_val = CelebA(
    root=DATA_ROOT, split="valid", target_type="attr",
    transform=TRANSFORMATION_DEFAULT
)

ds_val, _ = random_split(ds_val, [TRAINING_SIZE, 1-TRAINING_SIZE], generator=trng)

dl_val = DataLoader(
    ds_val,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,    
    shuffle=False,
)

In [7]:
class MultiTaskHead(nn.Module):
    def __init__(self, in_dims, num_tasks, out_per_task):
        super().__init__()
        
        arr_heads = []
        for tix in range(num_tasks):
            head = nn.Linear(in_features=in_dims, out_features=out_per_task)
            arr_heads.append(head)
        self.arr_heads = nn.ModuleList(arr_heads)
        self.task_id = None

    def forward(self, x):
        if self.task_id is not None:
            return self.arr_heads[self.task_id](x)
    
        arr_out = []

        
        for head in self.arr_heads:
            headout = head(x)
            b, d = headout.shape
            arr_out.append(headout.reshape(b, 1, d))

        out = torch.cat(arr_out, dim=1)
        return out
        
def get_state_dict(run_id):
    agent = wandb.Api()

    artifact: wandb.Artifact = agent.artifact(
        f"{WANDB_PROJECT}/model-{RUN_ID}:latest"
    )

    artifact_dir = artifact.download(root="/tmp")

    ckpt = torch.load(
        f"{artifact_dir}/model.ckpt",
        map_location=torch.device("cpu"),
        weights_only=False,
    )

    state_dict = ckpt["state_dict"]

    new_dict = OrderedDict()
    for k, v in state_dict.items():
        new_k = k.replace("encoder.", "")
        new_dict[new_k] = v
        
    return new_dict

def get_model(arch):

    state_dict = get_state_dict(RUN_ID)
    
    if arch == "resnet18":
        model = torchvision.models.resnet18(weights=None, num_classes=NUM_ATTRIBUTES)
        # out_dims, in_dims = model.fc.weight.shape
        
        # model.fc = MultiTaskHead(in_dims, NUM_ATTRIBUTES, 2)
    
    model.load_state_dict(state_dict)

    model.eval()
    model.to(DEVICE)
    
    return model
    
model = get_model(ARCH);

wandb: Downloading large artifact model-n8r0q2vb:latest, 128.27MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:0.6


In [8]:
def estimate_task_performance(model, dl, task_id, verbose=False):

    # metric = Accuracy(task="multiclass", num_classes=2)
    metric = BinaryAUROC(thresholds=20)
    model.to(DEVICE)

    for x, y in tqdm(dl, disable=not verbose):
        x = x.to(DEVICE)
        task_logit = model(x)[:, task_id].cpu()
        task_target = y[:, task_id]
        metric.update(task_logit, task_target)

    metric = float(metric.compute())
    metric = np.max([metric, 1-metric])
    return metric

# cross-check with wandb that we have correct results
estimate_task_performance(model, dl_val, task_id=0, verbose=True)

  0%|          | 0/32 [00:00<?, ?it/s]

0.9399108290672302

# Extract Activation and Context Vectors for Task

In [9]:
class VoidAttributor:

    def __enter__(self, **kwargs):
        pass

    def __exit__(self, type, value, tb):
        pass

def logodd(x):
    smg = F.sigmoid(x)
    logodd = torch.log(smg / (1-smg))
    assert torch.isfinite(logodd).all()
    return logodd

def extract_activation_context_for_task(
    model: nn.Module,
    layer: str,
    data_loader: DataLoader,
    task_id: int,
    use_lrp=True,
    seed=1,
    device=DEVICE,
    number_of_selected_spatial_locations=20,
    strict_mode=False,
    verbose=False
):
        
    arr_act = []
    arr_ctx = []

    rng = np.random.default_rng(seed=1)

    task_query_vector = F.one_hot(torch.tensor(task_id), num_classes=NUM_ATTRIBUTES).to(DEVICE)
    try:
        model.fc.task_id = task_id
        
        module, hook = utils.interceptor.attach_hook_intercept_layer_output(
            model, layer, should_retain_grad=True, detach_output=False
        )

        attributor = attributors.make_attributor_for(
            model,
            (
                TRANSFORMATION_DEFAULT.mean, TRANSFORMATION_DEFAULT.std
            )
        ) if use_lrp else VoidAttributor()
        
        with attributor:
            for batch in tqdm(data_loader, desc=f"[layer={layer}; use_lrp={use_lrp}] extract act ctx"):
                x, y = batch
                x = x.to(device)

                if use_lrp:
                    _ = attributor.forward(x, lambda logits: logits * task_query_vector)
    
                    act = utils.interceptor.get_output(module)
    
                    assert act.grad is not None
                    rel = act.grad
    
    
                    ctx = torch.where(act.abs() > 0, rel / act, 0)
                    assert torch.isfinite(ctx).all()
                    
                    if strict_mode:
                        np.testing.assert_allclose(
                            (act * ctx).detach().cpu().numpy(),
                            rel.detach().cpu().numpy(),
                            atol=1e-6,
                        )
                else:
                    logits = model(x)
                    (logits * task_query_vector).sum().backward()
                    act = utils.interceptor.get_output(module)
    
                    assert act.grad is not None
                    ctx = act.grad
                
                output_dimensions = act.shape[1:]


                assert ctx.shape == act.shape

                act = act.detach().cpu().numpy()
                ctx = ctx.detach().cpu().numpy()

                selected_act, selected_ctx = utils.subsample_tensors(
                    act,
                    ctx,
                    num_locations=number_of_selected_spatial_locations,
                    rng=rng,
                )
                arr_act.append(selected_act)
                arr_ctx.append(selected_ctx)

    finally:
        hook.remove()
        model.fc.task_id = None

    print(f"{layer}: output-dims={output_dimensions}")

    arr_act = np.vstack(arr_act)
    arr_ctx = np.vstack(arr_ctx)

    return arr_act, arr_ctx

def ano():

    for use_lrp in [True, False]:
            
    
        extract_activation_context_for_task(
            model, 
            "layer1",
            dl_train,
            task_id=0,
            use_lrp=use_lrp
        )
        
    print("Sanity check: passed!")
ano()

[layer=layer1; use_lrp=True] extract act ctx:   0%|          | 0/255 [00:00<?, ?it/s]

layer1: output-dims=torch.Size([64, 56, 56])


[layer=layer1; use_lrp=False] extract act ctx:   0%|          | 0/255 [00:00<?, ?it/s]

layer1: output-dims=torch.Size([64, 56, 56])
Sanity check: passed!


# Constructing Basis 

In [10]:
BASES = dict()

def register_basis():
    """Decorator to register a data modality provider."""

    def wrapped(cls):
        """Wrapped function to register a data modality provider with name `name`"""
        name = cls.__name__
        BASES[name] = cls

        return cls

    return wrapped

def _solve_eigvecs(cov, sort_func=lambda x: x):
    eigvals, eigvecs = np.linalg.eigh(cov)

    assert len(eigvals.shape) == 1

    indices = np.argsort(-sort_func(eigvals))
    eigvals = eigvals[indices]
    eigvecs = eigvecs[:, indices]

    return eigvecs

In [19]:
class BasisInterface:
    def get_Uk(self, k: int) -> npt.NDArray:
        raise NotImplementedError()   

@register_basis()
class PCABasis(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_act.T @ arr_act
        self.U = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

@register_basis()
class ContextPCABasis(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_ctx.T @ arr_ctx
        self.U = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

@register_basis()
class PRCASortAbsBasis(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U = _solve_eigvecs(cov, sort_func=np.abs)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]


@register_basis()
class PRCASignAlignSortAbsBasis(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        arr_rel = (arr_act * arr_ctx).sum(axis=1, keepdims=True)
        arr_ctx = (arr_rel >= 0) * arr_ctx - (arr_rel < 0) * arr_ctx
        
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        
        self.U = _solve_eigvecs(cov, sort_func=np.abs)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]


In [12]:
class PRCAReconNonGreedy:
    def __init__(self, task_id, layer):
        self.task_id = task_id
        self.layer = layer
    def fit(
        self, arr_act: npt.NDArray, arr_ctx: npt.NDArray, k: int,  U_init=None, device="cpu",
    ) -> npt.NDArray:
        n, d = arr_act.shape

        assert arr_ctx.shape == arr_act.shape, arr_ctx.shape
        
        lr = 1e-4
        epochs = 2000
                
        n, d,  = arr_act.shape

        arr_act = arr_act / ((np.mean(arr_act**2) ** (1 / 2)) * (d ** (1 / 4)))
        arr_ctx = arr_ctx / ((np.mean(arr_ctx**2) ** (1 / 2)) * (d ** (1 / 4)))
        
        arr_act: torch.Tensor = torch.from_numpy(arr_act).to(device)
        arr_ctx: torch.Tensor = torch.from_numpy(arr_ctx).to(device)


        linear_layer = torch.nn.Linear(k, d, bias=False)
        trng = torch.Generator()
        trng.manual_seed(1)
        if U_init is None:
            
            U_init = torch.randn((k, d), generator=trng)
        else:
            U_init = torch.from_numpy(U_init.T)
            
        linear_layer.weight = torch.nn.Parameter(U_init)
    
        ortho_layer = torch.nn.utils.parametrizations.orthogonal(linear_layer).to(device)
        assert ortho_layer.weight.shape == (k, d)
        
        optimizer = torch.optim.Adam(ortho_layer.parameters(), lr=lr)

        rel = (arr_act * arr_ctx).sum(dim=1)

        pgb = tqdm(range(epochs), desc=f"{self.__class__.__name__} (k={k})")
        for epoch in pgb:
            optimizer.zero_grad()
            
            # shape: (k, d)
            U = ortho_layer.weight

            act_proj = arr_act @ U.T

            ctx_proj = arr_ctx @ U.T
            assert act_proj.shape == (n, k), act_proj.shape
            rel_recon = (act_proj * ctx_proj).sum(dim=1)

            loss = ((rel - rel_recon)  ** 2).mean()
            
            loss.backward()
        
            optimizer.step()

            loss = loss.detach().cpu().numpy()
                
            pgb.set_description_str(f"{self.__class__.__name__} (k={k}) loss={loss:.4e}")
            
        U =  ortho_layer.weight.T.detach().cpu().numpy()
        
        # sanity_check
        np.testing.assert_allclose(U.T @ U, np.eye(k), atol=1e-4)

        return U


@register_basis()
class PRCAReconBasis(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        self.arr_act = arr_act
        self.arr_ctx = arr_ctx

        self.layer = layer
        self.task_id = task_id
        self.is_slow = True
        
    def get_Uk(self, k: int):

        return PRCAReconNonGreedy(
            layer=self.layer,
            task_id=self.task_id
        ).fit(
            self.arr_act, self.arr_ctx, 
            k=k,
            U_init=PRCASortAbsBasis(arr_act=self.arr_act, arr_ctx=self.arr_ctx).get_Uk(k),
            device=DEVICE
        )

In [13]:

# def estimate_basis(basis_name, arr_act, arr_ctx):
#     if basis_name == "pca":
#         cov = arr_act.T @ arr_act
#         eigvecs = _solve_eigvecs(cov)
#         return eigvecs
#     elif basis_name == "gpca":
#         cov = arr_ctx.T @ arr_ctx
#         eigvecs = _solve_eigvecs(cov)
#         return eigvecs
#     elif basis_name == "v0":
#         w = model.fc.weight[0, :].detach().cpu().numpy()
#         w = w / np.linalg.norm(w)
#         U = w.reshape((-1, 1))
#         return U
#     elif basis_name == "prca":
#         A = arr_act 
#         C = arr_ctx 
        
#         cov = A.T @ C + C.T @ A
#         eigvecs = _solve_eigvecs(cov)
#         return eigvecs
#     elif basis_name == "prca-sortabs":
#         cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
#         eigvecs = _solve_eigvecs(cov, sort_func=np.abs)
#         return eigvecs
#     elif basis_name == "prca-sign-align-sortabs":
#         arr_rel = (arr_act * arr_ctx).sum(axis=1, keepdims=True)
#         arr_ctx_signed = (arr_rel >= 0) * arr_ctx - (arr_rel < 0) * arr_ctx
#         return estimate_basis("prca-sortabs", arr_act, arr_ctx_signed)
#     elif basis_name == "prca-recon-greedy":
#         learner = PRCAReconGreedy()
#         return learner.fit(activation=arr_act, context=arr_ctx, device=DEVICE)
#     elif basis_name == "prca-sign-align-greedy-v3":
#         learner = PRCASignAlignGreedyV3()
#         return learner.fit(arr_act, arr_ctx, device=DEVICE)
#     elif "PRCAReconNonGreedy" in basis_name:
#         _, k = basis_name.split("k")
#         k = int(k)
#         return PRCAReconNonGreedy().fit(arr_act, arr_ctx, k=k, device=DEVICE)
#     else:
#         raise

# Estimating AUROCs

In [14]:
def construct_fh(Uk):
    def fh(mod, inp, outp):
        return F.conv2d(
            outp,
            (Uk@Uk.T).unsqueeze(2).unsqueeze(3)
        )
    return fh

def compute_task_aurocs_at_k(
    model, layer, task_id, 
    arr_ks,
    arr_ks_for_slow_learners,
    use_lrp,
    arr_basis_names=[
        "pca", 
        "prca-sortabs",
        "prca-sign-align-sortabs",
        # "gpca",
    ],
    base_output_dir=f"./artifacts/celeba-{RUN_ID}"
):
    arr_act, arr_ctx = extract_activation_context_for_task(
        model=model, 
        layer=layer, 
        data_loader=dl_train, 
        task_id=task_id,
        use_lrp=use_lrp
    )
    
    rel = (arr_act  * arr_ctx).sum(axis=1)

    module = getattr(model, layer)
    arr_ks = sorted(list(set(arr_ks + arr_ks_for_slow_learners)))

    suffix = "relevance-lrp" if use_lrp else "relevance-grad"
    output_path = f"{base_output_dir}/task-{task_id}/{layer}/{suffix}"
    os.makedirs(output_path, exist_ok=True)

    arr_dfs = []
    for basis_name in arr_basis_names:
        arr_stat_rows = []

        basis: BasisInterface = BASES[basis_name](
            arr_act=arr_act, 
            arr_ctx=arr_ctx,
            layer=layer,
            task_id=task_id
        )

        for k in tqdm(
            arr_ks_for_slow_learners if hasattr(basis, "is_slow") else arr_ks, 
            desc=f"[{basis_name:<20s}] Estimating Performance"
        ):
        
            if ("-k" in basis_name) and not f"{k}" == basis_name.split("k")[1]:
                continue

            Uk = basis.get_Uk(k=k)

        

            arr_recon_act = (arr_act @ Uk) @ Uk.T

            
            projected_rel = ((arr_act @ Uk) * (arr_ctx @ Uk)).sum(axis=1)
            
            assert rel.shape == projected_rel.shape == (rel.shape[0], )
            
            recon_err = np.linalg.norm(arr_act - arr_recon_act, axis=1).mean()
            rel_recon_err = ((rel - projected_rel) **2).mean()
            perc_sign_align = (np.sign(rel) * np.sign(projected_rel)).mean()

            row = dict(
                layer=layer,
                use_lrp=use_lrp,
                task_id=task_id, 
                k=k,
                basis_name=basis_name,
                recon_err=recon_err,
                rel_recon_err=rel_recon_err,
                perc_sign_align=perc_sign_align,
            )

            Uk = torch.from_numpy(Uk).to(DEVICE)

            try:
                hook = module.register_forward_hook(construct_fh(Uk))
                
                for label, dl in [
                    # ("train", dl_train_noshuffle),
                    ("val", dl_val)
                ]:
                    row[f"{label}_auroc"] = estimate_task_performance(model, dl, task_id)
                
            finally:
                hook.remove()
            arr_stat_rows.append(row)
            
     
        df = pd.DataFrame(arr_stat_rows)
        df.to_csv(
            f"{output_path}/{basis_name}.csv",
            index=False
        )

        arr_dfs.append(df)

    df = pd.concat(arr_dfs).sort_values(by=["k", f"val_auroc"], ascending=[True, False])
    print(f"Checking results at {output_path}")
    return df

compute_task_aurocs_at_k(
    model, layer="layer2", task_id=0, 
    arr_basis_names=[
        # "PCABasis", 
        # "ContextPCABasis",
        "PRCASortAbsBasis",
        "PRCAReconBasis",
    ],
    arr_ks=[10, 20, 30],
    arr_ks_for_slow_learners=[10, 20],
    use_lrp=False,
    base_output_dir="./tmp/celeba"
)

[layer=layer2; use_lrp=False] extract act ctx:   0%|          | 0/255 [00:00<?, ?it/s]

layer2: output-dims=torch.Size([128, 28, 28])


[PRCASortAbsBasis    ] Estimating Performance:   0%|          | 0/3 [00:00<?, ?it/s]

[PRCAReconBasis      ] Estimating Performance:   0%|          | 0/2 [00:00<?, ?it/s]

PRCAReconNonGreedy (k=10):   0%|          | 0/2000 [00:00<?, ?it/s]

PRCAReconNonGreedy (k=20):   0%|          | 0/2000 [00:00<?, ?it/s]

Checking results at ./tmp/celeba/task-0/layer2/relevance-grad


,layer,use_lrp,task_id,k,basis_name,recon_err,rel_recon_err,perc_sign_align,val_auroc
0,layer2,False,0,10,PRCASortAbsBasis,10.164715,0.001617,0.438508,0.932428
0,layer2,False,0,10,PRCAReconBasis,10.658664,0.000687,0.629127,0.914873
1,layer2,False,0,20,PRCASortAbsBasis,8.990715,0.000777,0.620722,0.958271
1,layer2,False,0,20,PRCAReconBasis,9.064050,0.000304,0.746937,0.947779
2,layer2,False,0,30,PRCASortAbsBasis,8.097192,0.000401,0.723518,0.958962


# Getting Results

In [16]:
ARR_LAYER_DIMENSIONS = utils.get_dimensions_at_layers(
    model=model,
    dataloader=dl_train,
    layers=["layer1", "layer2", "layer3", "layer4"],
    device=DEVICE
)

In [17]:
ARR_LAYER_DIMENSIONS

{'layer1': 64, 'layer2': 128, 'layer3': 256, 'layer4': 512}

In [20]:
for use_lrp in [False, True]:
    
    for layer in tqdm(["layer1", "layer2", "layer3", "layer4"], desc=f"use_lrp={use_lrp}"):
        d = ARR_LAYER_DIMENSIONS[layer]
        
        arr_ks = sorted(set(
            np.arange(1, 10).tolist() + 
            np.linspace(1, d, 8).astype(int).tolist()
        ))

        arr_ks_slow_learners = [1, 10, 20, 30, 40, 50]

        for task_id in [0, 25]:
                
            compute_task_aurocs_at_k(
                model, layer=layer, task_id=task_id, 
                arr_basis_names=[
                    # "PCABasis", 
                    # "ContextPCABasis",
                    # "PRCASortAbsBasis",
                    # "PRCAReconBasis",
                    "PRCASignAlignSortAbsBasis",
                ],
                arr_ks=arr_ks,
                arr_ks_for_slow_learners=arr_ks_slow_learners,
                use_lrp=use_lrp,
            )

use_lrp=False:   0%|          | 0/4 [00:00<?, ?it/s]

[layer=layer1; use_lrp=False] extract act ctx:   0%|          | 0/255 [00:00<?, ?it/s]

layer1: output-dims=torch.Size([64, 56, 56])


[PRCASignAlignSortAbsBasis] Estimating Performance:   0%|          | 0/20 [00:00<?, ?it/s]

Checking results at ./artifacts/celeba-n8r0q2vb/task-0/layer1/relevance-grad


...

Checking results at ./artifacts/celeba-n8r0q2vb/task-0/layer4/relevance-lrp


[layer=layer4; use_lrp=True] extract act ctx:   0%|          | 0/255 [00:00<?, ?it/s]

layer4: output-dims=torch.Size([512, 7, 7])


[PRCASignAlignSortAbsBasis] Estimating Performance:   0%|          | 0/21 [00:00<?, ?it/s]

Checking results at ./artifacts/celeba-n8r0q2vb/task-25/layer4/relevance-lrp


In [ ]:
raise

In [ ]:
def getting_aurocs_for_tasks(task_id, arr_layers, use_lrp,
    arr_basis_names=["pca"]
                            ):
    arr_layer_dims = utils.get_dimensions_at_layers(
        model=model,
        dataloader=dl_train,
        layers=arr_layers,
        device=DEVICE
    )

    arr_dfs = []

    for layer in tqdm(arr_layers, desc="getting results for layer"):
        d = arr_layer_dims[layer]
        arr_ks = sorted(set(
            np.arange(1, 10).tolist() + 
            np.linspace(1, d, 8).astype(int).tolist()
        ))
        df = compute_task_aurocs_at_k(
            model, 
            layer=layer, 
            task_id=task_id, 
            arr_ks=arr_ks,
            use_lrp=use_lrp,
            arr_basis_names=arr_basis_names
        )
        
        arr_dfs.append(df)

    return pd.concat(arr_dfs)

df = getting_aurocs_for_tasks(
    task_id=0,
    arr_layers=["layer1", "layer2", "layer3", "layer4"],
    use_lrp=True,
    arr_basis_names=[
        "pca", 
        "prca-sortabs",
        "prca-recon-greedy",
        "prca-sign-align-greedy-v3",
    ]
)

In [ ]:
df

In [ ]:
def viz_results(df):


    task_id = df.task_id.values[-1]
    arr_layers = df.layer.unique()
    ncols = len(arr_layers)

    use_lrp = df.use_lrp.values[-1]


    arr_basis_names = df.basis_name.unique()
    plt.figure(figsize=(3*ncols, 2))
    plt.suptitle(f"arch={ARCH} Task {task_id} (run_id={RUN_ID}, use_lrp={use_lrp}, ts={TRAINING_SIZE})", y=1.1)
    metric_col_name = "val_auroc"
    for lix, layer in enumerate(arr_layers):
        plt.subplot(1, ncols, lix + 1)
        plt.title(f"Layer {layer}")

        for bix, basis_name in enumerate(arr_basis_names):
            
            _df = df[
                (df.task_id == task_id) & (df.basis_name == basis_name) & (df.layer == layer)
            ]
            if bix == 0:
                plt.axhline(
                    _df[metric_col_name].values[-1],
                    ls="--",
                    lw=1, 
                    color="k"
                )
            plt.plot(
                _df.k,
                _df[metric_col_name],
                label=basis_name,
                color="black" if basis_name == "pca" else None
            )

            d = _df.k.values[-1]
            
        plt.ylim([0.5, 1.0])
        if lix == 0:
            plt.ylabel("AUROC (val)")
            plt.legend()
        plt.xlabel(f"Subspace Dimensions K\n(d={d})")

viz_results(df)

In [ ]:
# viz_results(getting_aurocs_for_tasks(
#     task_id=25,
#     arr_layers=["layer1", "layer2", "layer3", "layer4"],
#     use_lrp=True
# ))

In [ ]:
# viz_results(getting_aurocs_for_tasks(
#     task_id=0,
#     arr_layers=["layer1", "layer2", "layer3", "layer4"],
#     use_lrp=False
# ))

In [ ]:
# viz_results(getting_aurocs_for_tasks(
#     task_id=25,
#     arr_layers=["layer1", "layer2", "layer3", "layer4"],
#     use_lrp=False
# ))

In [ ]:
# for task_id in [10, 20, 30]:
#     viz_results(getting_aurocs_for_tasks(
#         task_id=task_id,
#         arr_layers=["layer1", "layer2", "layer3", "layer4"],
#         use_lrp=True
#     ))

In [ ]:
print(f"finished at {datetime.now()}")

# Dev Notes
## 2024/12/17
- [x] get act/ctx
- [x] basis
- [x] forwardhook